# Week 2.2 | Which EGFR inhibitors would you compare in 3D?

In Week 1, you found activity data for gefitinib and osimertinib in ChEMBL. In Week 2.1, you selected three EGFR inhibitors using drawings and fingerprint similarity. Today you will test that selection with **clustering**, then look at experimental structures to see what the ligands share in the EGFR pocket.

**Research question.** How would you use 2D structural diversity to choose three inhibitors for a 3D comparison?

### Learning objectives

- Read a Tanimoto distance matrix and check that the numbers behave as expected.
- Use Butina clustering to group inhibitors and explain how the cutoff changes the groups.
- Justify a three-drug shortlist using cluster coverage and pairwise scores.
- Compare a shared contact and a difference in how two inhibitors sit in the EGFR kinase domain.

### Plan · 3 hours

1. Reopen your Week 2.1 shortlist · 15 min
2. Read and check a distance matrix · 30 min
3. Group the inhibitors · 40 min
4. Revisit your shortlist · 25 min
5. Check the experimental structures · 40 min
6. Discuss your decision · 15 min

Take a 15-minute break after Section 3.

**How to work.** Predict a result, run the code, and check it against a table, drawing or source. Use the optional GenAI prompts to explain code if you find them helpful. You can do the same checks with a partner. Record what you checked and what the evidence showed.

## 1. Reopen your Week 2.1 shortlist

We use the same data from the course GitHub repository. Run the setup and loading cells in order. Record the three IDs you chose in Week 2.1. If you need a starting set, use D01, D02 and D05.

**Optional GenAI prompt**

> Explain what RDKit is used for in this practical.

In [ ]:
# @title Install RDKit { display-mode: "form" }
%pip -q install rdkit

This cell makes the tools for tables, drawings and clustering available.

**Optional GenAI prompt**

> Explain what pandas and RDKit do here in two sentences.

In [ ]:
# @title Import the packages { display-mode: "form" }
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from itertools import combinations

from IPython.display import display
from rdkit import Chem, DataStructs
from rdkit.Chem import Draw, MACCSkeys, rdFingerprintGenerator
from rdkit.ML.Cluster import Butina

`read_csv` reads the compound table from GitHub. The course IDs become the row labels. The `reference` column marks the six drugs we use in the main exercise.

**Optional GenAI prompt**

> Explain how the code selects six rows from the full table.

In [ ]:
data_url = "https://raw.githubusercontent.com/tcobanoglu/dadd-course-data/main/data/week2/egfr_compounds.csv"
collection = pd.read_csv(data_url, index_col="compound_id")
drugs = collection.loc[collection["reference"]].copy()

drugs[["name", "chembl_id", "pdb_id"]]

RDKit reads the SMILES for each drug and draws the six structures. Look for rings, branching and the positions of labelled atoms.

**Predict.** Which pair do you expect to be closest with the Week 2.1 Morgan settings? Point to one shared feature before you see the scores.

**Optional GenAI prompt**

> Explain how `.map` turns the six SMILES into six RDKit molecules.

In [ ]:
molecules = drugs["smiles"].map(Chem.MolFromSmiles)
labels = (drugs.index + " " + drugs["name"]).tolist()

Draw.MolsToGridImage(molecules.tolist(), legends=labels, molsPerRow=3, subImgSize=(250, 170))

**My starting shortlist, predicted pair and reason**

_Write here._

## 2. Read and check a distance matrix

In Week 2.1, a higher Tanimoto score meant that two fingerprints shared a greater fraction of their on-bits. For clustering, we work with **distance**. Here, the distance is **1 minus Tanimoto similarity**. More similar fingerprints have smaller distances.

Run the first cell to calculate the same binary Morgan and MACCS fingerprints as Week 2.1.

**Optional GenAI prompt**

> Explain why there is one fingerprint per drug for each method.

In [ ]:
morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
morgan_fingerprints = molecules.map(morgan.GetFingerprint)
maccs_fingerprints = molecules.map(MACCSkeys.GenMACCSKeys)

The function compares each drug's fingerprint with all six, including itself. The resulting table has the same IDs on each axis. Run the cell and find your predicted pair.

**Optional GenAI prompt**

> Explain what one row and the diagonal show in the similarity table.

In [ ]:
# @title Compare each fingerprint with the other fingerprints { display-mode: "form" }
def similarity_table(fingerprints):
    all_fingerprints = fingerprints.tolist()
    rows = []
    for fingerprint in all_fingerprints:
        scores = DataStructs.BulkTanimotoSimilarity(fingerprint, all_fingerprints)
        rows.append(scores)
    return pd.DataFrame(rows, index=fingerprints.index, columns=fingerprints.index)

morgan_similarity = similarity_table(morgan_fingerprints)
maccs_similarity = similarity_table(maccs_fingerprints)

morgan_similarity.round(3)

### Code review · Did the script calculate a distance?

Your colleague claims the marked line makes a distance table. `.copy()` makes a copy of a table.

**Predict.** What distance should a drug have to itself? Run the cell as written. Check a diagonal value, then change only the marked line. Verify that the diagonal now matches your prediction. Compare the value for D01–D06 with the previous similarity table.

**Optional GenAI prompt**

> Explain what the marked line does to the values and suggest a one-line correction.

In [ ]:
review_distance = morgan_similarity.copy()  # Review this line.

print("D05 to itself:", review_distance.loc["D05", "D05"])
print("D01 to D06:", round(review_distance.loc["D01", "D06"], 3))

**My prediction, the line I changed and the value I checked**

_Write here._

Use the verified distance matrices below for clustering. A **heatmap** shows the same numbers as a coloured grid. Read the scale beside the grid. Which pair of different drugs has the smallest Morgan distance? Does it match your prediction?

**Optional GenAI prompt**

> Explain how this heatmap can show the closest pair and why its diagonal is zero.

In [ ]:
# @title Plot the checked distance matrix { display-mode: "form" }
morgan_distance = 1 - morgan_similarity
maccs_distance = 1 - maccs_similarity
distance_tables = {"Morgan": morgan_distance, "MACCS": maccs_distance}

plt.figure(figsize=(6.5, 5))
sns.heatmap(morgan_distance, annot=True, fmt=".3f", cmap="Blues", vmin=0, vmax=1)
plt.title("Morgan fingerprint distance")
plt.show()

## 3. Group the inhibitors with Butina

A **cluster** is a group of compounds selected using a rule. In **Butina clustering**, count how many neighbours each compound has within a chosen distance. Start with the compound with the most neighbours as a **centre**. Group it with its available neighbours, then continue down the list of candidates. The centre is one of the actual drugs. A **cutoff** of 0.60 means a candidate can be grouped with that centre if its distance from the centre is at most 0.60 and it is still available.

This is the clustering method used in TeachOpenCADD T005. Here we begin with the same Morgan fingerprints you used in Week 2.1. [TeachOpenCADD T005](https://projects.volkamerlab.org/teachopencadd/talktorials/T005_compound_clustering.html)

Run the preparation cell, then work with the two settings below it.

**Optional GenAI prompt**

> Explain how the function turns Butina's positions into course IDs and identifies the centre.

In [ ]:
# @title Prepare Butina clustering { display-mode: "form" }
def cluster_table(distances, cutoff):
    positions = Butina.ClusterData(
        distances.to_numpy(), len(distances), cutoff, isDistData=True
    )

    rows = []
    for number, group in enumerate(positions, start=1):
        group_ids = distances.index[list(group)]
        centre_id = group_ids[0]
        for drug_id in group_ids:
            rows.append([drug_id, number, centre_id])

    columns = ["compound_id", "Cluster", "Centre"]
    return pd.DataFrame(rows, columns=columns).set_index("compound_id")

**Predict.** Will a smaller cutoff allow more or fewer neighbours around a centre?

**Run and check.**

1. Use Morgan and start at `cutoff = 0.60`. Count the groups and record each centre.
2. Change only the cutoff to `0.45`. Which pair still groups together? Which drugs are alone?
3. Change the fingerprint to `"MACCS"`. Try `0.60` and then `0.30`. Compare the three-group MACCS result with the three-group Morgan result. Does either drug move when you change the representation?

Compare membership by IDs. The cluster number is just a label.

**Optional GenAI prompt**

> Explain what changing the cutoff does and interpret one row of the table.

In [ ]:
fingerprint_type = "Morgan"
cutoff = 0.60

distances = distance_tables[fingerprint_type]
groups = cluster_table(distances, cutoff)
display(groups.join(drugs[["name"]]))
print("Number of groups:", groups["Cluster"].nunique())

### Check what a group means

In the Morgan result at 0.60, follow afatinib (D01) and erlotinib (D03). Run this cell with the three IDs shown. Compare each drug's distance to D05 with the distance between D01 and D03. Use this evidence to explain whether all pairs in a Butina cluster have to meet the cutoff.

**Optional GenAI prompt**

> Explain how using the same list on both axes selects three drugs from the matrix.

In [ ]:
group_ids = ["D05", "D01", "D03"]
group_distances = morgan_distance.loc[group_ids, group_ids]

group_distances.round(3)

**My groups, one change I observed and what I checked**

_Write here._

### Break · 15 minutes

## 4. Revisit your shortlist

You can compare three complexes. One rule is to select **one member per group**. A second rule is to avoid choosing a pair that looks especially similar by your fingerprint. These are two ways to use the data to make a decision.

The cell below restores the three Morgan groups at 0.60. The group centres give you a first candidate set. You can choose a different member from a multi-drug group.

**Optional GenAI prompt**

> Explain how the table identifies one centre per group.

In [ ]:
shortlist_groups = cluster_table(morgan_distance, cutoff=0.60)
display(shortlist_groups.join(drugs[["name"]]))

centres = shortlist_groups["Centre"].drop_duplicates().tolist()
print("One possible shortlist:", centres)

**Edit and run.** Put your three Week 2.1 IDs in the first list and your proposed three IDs in the second. Compare the number of groups each set covers and the highest Morgan similarity within each set. The function lists each distinct pair once.

**Discuss.** Did a change improve one of these criteria while reducing the other? Choose three drugs and explain which criterion mattered more for your decision.

**Optional GenAI prompt**

> Explain how the code counts represented groups and finds the highest similarity between different drugs.

In [ ]:
original_ids = ["D01", "D02", "D05"]
selected_ids = ["D05", "D02", "D04"]

def show_shortlist(ids):
    assert len(set(ids)) == len(ids) == 3, "Use three different course IDs."
    selected = shortlist_groups.loc[ids]
    print("Morgan groups covered:", selected["Cluster"].nunique())

    rows = []
    for first, second in combinations(ids, 2):
        score = morgan_similarity.loc[first, second]
        rows.append([first, second, score])

    scores = pd.DataFrame(rows, columns=["First drug", "Second drug", "Similarity"])
    display(scores.round(3))
    print("Highest pair similarity:", round(scores["Similarity"].max(), 4))

print("Week 2.1 shortlist")
show_shortlist(original_ids)
print("Week 2.2 shortlist")
show_shortlist(selected_ids)

**My three IDs and why I chose them**

_Write here._

**One feature I predict two of my chosen drugs will share in the pocket**

_Write here._

## 5. Check the experimental structures

EGFR inhibitors can share a region of the pocket even when their fingerprints differ. We will compare one point on the **hinge**, a short part of the kinase that borders the ATP-binding pocket. Then we will look at the rest of each ligand in an overlay.

### Compare a contact

The coordinate table contains two atoms per structure. One is the backbone nitrogen of EGFR Met793. The other is the closest modelled nitrogen in that inhibitor. The 1M17 file numbers this same residue as Met769. Each row contains coordinates from one deposited complex, in ångströms.

**Run and inspect.** Find the PDB entry, chain, residue and atom labels for D05 and D03. Use the labels to locate the relevant atoms in the overlay.

**Optional GenAI prompt**

> Explain how the same course IDs link the compound and coordinate tables.

In [ ]:
coords_url = "https://raw.githubusercontent.com/tcobanoglu/dadd-course-data/main/data/week2/egfr_hinge_atoms.csv"
hinge_atoms = pd.read_csv(coords_url, index_col="compound_id")
atom_labels = ["pdb_id", "chain", "ligand_code", "ligand_residue", "ligand_atom", "protein_residue", "protein_atom"]

hinge_atoms.loc[["D05", "D03"], atom_labels]

**Predict.** Do you expect the two ligands to place their nitrogen at a similar distance from the hinge?

**Edit and run.** Start with D05 and D03. Change the second ID to D04. Then try two from your own shortlist. The numbers are lengths in ångströms. For each measurement the two atoms must share a coordinate frame.

A short distance can suggest a contact. To identify the interaction, inspect the atom types and their geometry.

**Optional GenAI prompt**

> Explain what the two lists of coordinates represent and what `dist` returns.

In [ ]:
from math import dist

def hinge_distance(drug_id):
    row = hinge_atoms.loc[drug_id]
    ligand_xyz = row[["ligand_x", "ligand_y", "ligand_z"]]
    protein_xyz = row[["protein_x", "protein_y", "protein_z"]]
    return dist(ligand_xyz, protein_xyz)

first_id = "D05"
second_id = "D03"

print(first_id, round(hinge_distance(first_id), 2), "Å")
print(second_id, round(hinge_distance(second_id), 2), "Å")

### Compare the rest of the pose

For the figure below, the protein kinase domains were aligned using matching residues near the hinge. This puts the ligands in a shared frame. The views use the same orientation. The labelled hinge residue in each panel comes from the gefitinib structure. TeachOpenCADD T008 also prepares protein–ligand structures for visual comparison. [TeachOpenCADD T008](https://projects.volkamerlab.org/teachopencadd/talktorials/T008_query_pdb.html)

![Two coordinate overlays comparing gefitinib with erlotinib and lapatinib in the EGFR pocket](https://raw.githubusercontent.com/tcobanoglu/dadd-course-data/main/data/week2/egfr_binding_overlay.webp)

**Discuss.** Follow the ligands outward from the hinge. Where do they overlap? Where does lapatinib extend relative to gefitinib? Explain how both drugs can be close to the hinge while occupying different parts of the pocket.

For a pair from your own shortlist, locate each complex using its PDB ID in the table below. Start with the 3D View. Describe one feature you can check from the models.

**Optional GenAI prompt**

> Explain how these two lines turn each PDB entry ID into a structure link.

In [ ]:
for drug_id in selected_ids:
    pdb_id = drugs.loc[drug_id, "pdb_id"]
    print(drug_id, "https://www.rcsb.org/structure/" + pdb_id)

**Two atom distances, one shared feature and one difference**

_Write here._

## 6. Discuss your decision

1. **Clustering.** Which shortlist would you use? Report the three IDs, which groups they cover and one pairwise score that informed your choice.
2. **Structure.** Give one example of a feature shared in the pocket and one difference in how a drug extends through it. Identify the experimental structures or the overlay you used.
3. **Week 1 link.** What can an IC₅₀ value tell you? What can the structure tell you? Name one assay detail you would check before comparing two IC₅₀ values.
4. **Evidence check.** Record one prediction you tested, the result you used to check it and what you would change. If you used GenAI, include which part of its explanation you verified.

**My final decision and the evidence I used**

_Write here._

## Optional · Try the fingerprint used in TeachOpenCADD T005

T005 uses an RDKit path fingerprint. It records patterns along paths of bonds through the molecule. Change the cutoff from 0.45 to 0.60. Compare the number of groups with the Morgan results and explain why the same threshold can behave differently.

**Optional GenAI prompt**

> Explain which line changes the fingerprint and which lines reuse the clustering method.

In [ ]:
path_generator = rdFingerprintGenerator.GetRDKitFPGenerator(maxPath=5)
path_fingerprints = molecules.map(path_generator.GetFingerprint)
path_distance = 1 - similarity_table(path_fingerprints)

path_cutoff = 0.45
path_groups = cluster_table(path_distance, path_cutoff)
path_groups.join(drugs[["name"]])

## Optional · Compare the full 28-compound set

Keep the same Morgan settings and 0.60 cutoff. Predict whether adding 22 compounds could change the cluster centres of the reference drugs. Run the cell. Compare the centres with Section 3 and count how many compounds are in each group.

**Optional GenAI prompt**

> Explain why the code prints all group sizes but shows only six drugs in the table.

In [ ]:
all_molecules = collection["smiles"].map(Chem.MolFromSmiles)
all_fingerprints = all_molecules.map(morgan.GetFingerprint)
all_distances = 1 - similarity_table(all_fingerprints)

all_groups = cluster_table(all_distances, cutoff=0.60)
display(all_groups.groupby("Centre")["Cluster"].size())
all_groups.loc[drugs.index]

## References and data

- [Course compound CSV](https://github.com/tcobanoglu/dadd-course-data/blob/main/data/week2/egfr_compounds.csv) and [coordinate CSV](https://github.com/tcobanoglu/dadd-course-data/blob/main/data/week2/egfr_hinge_atoms.csv).
- [TeachOpenCADD T004 · Compound similarity](https://projects.volkamerlab.org/teachopencadd/talktorials/T004_compound_similarity.html), [T005 · Compound clustering](https://projects.volkamerlab.org/teachopencadd/talktorials/T005_compound_clustering.html), and [T008 · Protein data acquisition](https://projects.volkamerlab.org/teachopencadd/talktorials/T008_query_pdb.html).
- [RCSB 2ITY · Gefitinib](https://www.rcsb.org/structure/2ITY), [1M17 · Erlotinib](https://www.rcsb.org/structure/1M17), and [1XKK · Lapatinib](https://www.rcsb.org/structure/1XKK).
- [Wood et al. (2004)](https://doi.org/10.1158/0008-5472.CAN-04-1168). The paper compares the EGFR kinase conformations observed with erlotinib and lapatinib.
- [AI-Guided Inquiry Learning](https://github.com/dayanjan/AGIL-VIP-AI-PSCI-Public) informed the predict, run, check and discuss exercises.